In [33]:
SELECT count(*) as number_of_tables
FROM information_schema.tables
WHERE table_type = 'BASE TABLE'

(1 row affected)

number_of_tables
----------------
30              
(1 row)

Total execution time: 00:00:00.215

To undestand our analysis we find out we have a total of 30 tables in our database, these are already organized in fact and dimension tables

In [39]:
select count(*) as B2C_ROWS
from factinternetsales;
GO
select count(*) as B2B_ROWS
from factresellersales;

(1 row affected)

B2C_ROWS
--------
60398   
(1 row)

(1 row affected)

B2B_ROWS
--------
60855   
(1 row)

Total execution time: 00:00:00.483

Our analysis will moslty be based on this two columsns and as we can see, We have a total of around 120k records in total.

In [54]:
select min(orderdate) as first_order_date,max(orderdate) as last_order_date
from factresellersales
UNION ALL
select min(orderdate) as first_order_date,max(orderdate) as last_order_date
from factinternetsales
;

(2 rows affected)

first_order_date        | last_order_date        
------------------------+------------------------
2010-12-29 00:00:00.000 | 2013-11-29 00:00:00.000
2010-12-29 00:00:00.000 | 2014-01-28 00:00:00.000
(2 rows)

Total execution time: 00:00:01.027

And this is the current date range for our fact tables. 

In [53]:
drop view if exists Orders_by_category;
GO
CREATE VIEW Orders_by_category AS
    select englishproductcategoryname as Category,englishproductsubcategoryname AS SubCategory,sum(OrderQuantity) as number_of_orders,sum(ExtendedAmount) as Revenue, sum(ExtendedAmount - TotalProductCost) as profit
    from factinternetsales
    LEFT JOIN dimproduct
    ON factinternetsales.productkey = dimproduct.ProductKey
    left join dimproductsubcategory
    on dimproduct.ProductSubcategoryKey = dimproductsubcategory.ProductSubcategoryKey
    left join dimproductcategory
    on dimproductsubcategory.productcategorykey = dimproductcategory.productcategorykey
    --where dimproduct.ProductSubcategoryKey is not null 
    and dimproductcategory.englishproductcategoryname is not null
    group by englishproductsubcategoryname,englishproductcategoryname;
    --group by englishproductcategoryname;
go
select Category,SubCategory,number_of_orders
from Orders_by_category
order by number_of_orders DESC;

Commands completed successfully.

Commands completed successfully.

(17 rows affected)

Category    | SubCategory       | number_of_orders
------------+-------------------+-----------------
Accessories | Tires and Tubes   | 17332           
Bikes       | Road Bikes        | 8068            
Accessories | Bottles and Cages | 7981            
Accessories | Helmets           | 6440            
Bikes       | Mountain Bikes    | 4970            
Clothing    | Jerseys           | 3332            
Clothing    | Caps              | 2190            
Bikes       | Touring Bikes     | 2167            
Accessories | Fenders           | 2121            
Clothing    | Gloves            | 1430            
Clothing    | Shorts            | 1019            
Accessories | Cleaners          | 908             
Accessories | Hydration Packs   | 733             
Clothing    | Socks             | 568             
Clothing    | Vests             | 562             
Accessories | Bike Racks        | 328        

here we can see, where our sales are coming from based categories and subcategories

In [50]:
select category, subcategory, revenue
from Orders_by_category
order by revenue DESC

(17 rows affected)

category    | subcategory       | revenue    
------------+-------------------+------------
Bikes       | Road Bikes        | 14520584.00
Bikes       | Mountain Bikes    | 9952760.00 
Bikes       | Touring Bikes     | 3844801.00 
Accessories | Tires and Tubes   | 245529.00  
Accessories | Helmets           | 225336.00  
Clothing    | Jerseys           | 172951.00  
Clothing    | Shorts            | 71320.00   
Accessories | Bottles and Cages | 56798.00   
Accessories | Fenders           | 46620.00   
Accessories | Hydration Packs   | 40308.00   
Accessories | Bike Stands       | 39591.00   
Accessories | Bike Racks        | 39360.00   
Clothing    | Vests             | 35687.00   
Clothing    | Gloves            | 35021.00   
Clothing    | Caps              | 19688.00   
Accessories | Cleaners          | 7219.00    
Clothing    | Socks             | 5106.00    
(17 rows)

Total execution time: 00:00:00.239

In [48]:
select category, subcategory, profit
from Orders_by_category
order by profit DESC

(17 rows affected)

category    | subcategory       | profit      
------------+-------------------+-------------
Bikes       | Road Bikes        | 5537299.6986
Bikes       | Mountain Bikes    | 4513624.1061
Bikes       | Touring Bikes     | 1454872.6959
Accessories | Tires and Tubes   | 153700.7512 
Accessories | Helmets           | 141059.828  
Clothing    | Shorts            | 44646.1603  
Clothing    | Jerseys           | 39778.6564  
Accessories | Bottles and Cages | 35555.3477  
Accessories | Fenders           | 29183.8995  
Accessories | Hydration Packs   | 25232.5721  
Accessories | Bike Stands       | 24783.966   
Accessories | Bike Racks        | 24639.36    
Clothing    | Vests             | 22340.062   
Clothing    | Gloves            | 21922.901   
Clothing    | Caps              | 4528.263    
Accessories | Cleaners          | 4518.8436   
Clothing    | Socks             | 3196.5336   
(17 rows)

Total execution time: 00:00:00.672

In [46]:
WITH sales_per_year AS (
    SELECT 
        YEAR(orderdate) AS order_year, 
        SUM(orderquantity) AS total_orders
    FROM factinternetsales
    GROUP BY YEAR(orderdate)
)
SELECT 
    total_orders,
    order_year,
   ROUND(
    (
        total_orders - LAG(total_orders) OVER (ORDER BY order_year ASC)
    )
    / CAST(LAG(total_orders) OVER (ORDER BY order_year ASC) AS DECIMAL(10,2))
    * 100 ,0) AS percentage_change
FROM sales_per_year;

GO

WITH b2b_sales_per_year AS (
    SELECT 
        YEAR(orderdate) AS order_year, 
        SUM(orderquantity) AS total_orders
    FROM factresellersales
    GROUP BY YEAR(orderdate)
)
SELECT 
    total_orders,
    order_year,
   ROUND(
    (
        total_orders - LAG(total_orders) OVER (ORDER BY order_year ASC)
    )
    / CAST(LAG(total_orders) OVER (ORDER BY order_year ASC) AS DECIMAL(10,2))
    * 100 ,0) AS percentage_change
FROM b2b_sales_per_year;

In [53]:
select column_name
from information_schema.columns
where table_name = 'dimproduct';

(36 rows affected)

column_name          
---------------------
ProductKey           
ProductAlternateKey  
ProductSubcategoryKey
WeightUnitMeasureCode
SizeUnitMeasureCode  
EnglishProductName   
SpanishProductName   
FrenchProductName    
StandardCost         
FinishedGoodsFlag    
Color                
SafetyStockLevel     
ReorderPoint         
ListPrice            
Size                 
SizeRange            
Weight               
DaysToManufacture    
ProductLine          
DealerPrice          
Class                
Style                
ModelName            
LargePhoto           
EnglishDescription   
FrenchDescription    
ChineseDescription   
ArabicDescription    
HebrewDescription    
ThaiDescription      
GermanDescription    
JapaneseDescription  
TurkishDescription   
StartDate            
EndDate              
Status               
(36 rows)

Total execution time: 00:00:00.278

In [30]:
with top_10_orders as 
        (select top 10 EnglishProductName as Top_10_by_order, rank() over(order by sum(OrderQuantity) DESC) AS ranking
        FROM factinternetsales as fs
        LEFT JOIN dimproduct as dp on fs.ProductKey = dp.ProductKey
        group by EnglishProductName),
    
        
top_10_revenue as (select top 10 EnglishProductName as Top_10_by_revenue, rank() over(order by sum(ExtendedAmount) DESC) AS ranking
        FROM factinternetsales as fs
        LEFT JOIN dimproduct as dp on fs.ProductKey = dp.ProductKey
        group by EnglishProductName),


top_10_profit as (select top 10 EnglishProductName as Top_10_by_profit, rank() over(order by sum(ExtendedAmount - TotalProductCost) DESC) AS ranking
        FROM factinternetsales as fs
        LEFT JOIN dimproduct as dp on fs.ProductKey = dp.ProductKey
        group by EnglishProductName)
        


select Top_10_by_order,Top_10_by_revenue,Top_10_by_profit
from top_10_orders 
LEFT JOIN top_10_revenue on top_10_orders.ranking = top_10_revenue.ranking
LEFT JOIN top_10_profit on top_10_orders.ranking = top_10_profit.ranking
;

(10 rows affected)

Top_10_by_order         | Top_10_by_revenue       | Top_10_by_profit       
------------------------+-------------------------+------------------------
Water Bottle - 30 oz.   | Mountain-200 Black, 46  | Mountain-200 Black, 46 
Patch Kit/8 Patches     | Mountain-200 Black, 42  | Mountain-200 Black, 42 
Mountain Tire Tube      | Mountain-200 Silver, 38 | Mountain-200 Silver, 38
Road Tire Tube          | Mountain-200 Silver, 46 | Mountain-200 Silver, 46
Sport-100 Helmet, Red   | Mountain-200 Black, 38  | Mountain-200 Black, 38 
AWC Logo Cap            | Mountain-200 Silver, 42 | Mountain-200 Silver, 42
Sport-100 Helmet, Blue  | Road-150 Red, 48        | Road-150 Red, 48       
Fender Set - Mountain   | Road-150 Red, 62        | Road-150 Red, 62       
Sport-100 Helmet, Black | Road-150 Red, 52        | Road-150 Red, 52       
Mountain Bottle Cage    | Road-150 Red, 56        | Road-150 Red, 56       
(10 rows)

Total execution time: 00:00:03.160

Here we can find some interesting insights, the products that are sold the most are accesories, but the bikes are the ones that accounts for the most profit and revenue, and theres is also a really strong correlation between revenue and profit. and for the top 10 products is an identical correlation. This is an analysis for the B2C or Internet sales. 

In [58]:
select avg(ExtendedAmount) as AOV_B2C
from FactInternetSales;
GO
SELECT AVG(extendedAmount) as AOV_B2B
FROM FactResellerSales;

(1 row affected)

AOV_B2C 
--------
486.0869
(1 row)

(1 row affected)

AOV_B2B  
---------
1330.6729
(1 row)

Total execution time: 00:00:00.387

The Average Order Per Customer is significantly higher on the b2b channel as compared to the b2c channel as expected.